In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
from langchain_community.document_loaders import HuggingFaceDatasetLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from src.config import DATASET_NAME, PAGE_CONTENT_COLUMN

from src.vector_store import create_vector_store
from src.data_acquisition import prepare_data
from src.llm import llm

C:\Users\Admin\AppData\Local\Temp\ipykernel_17820\2407047613.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import HuggingFaceDatasetLoader
C:\Users\Admin\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Admin\Desktop\Data Science\проекти\проект №4\src\llm.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai

TESTING CHUNK_SIZE AND CHUNK_OVERLAP VALUES

In [3]:
# data acquisition

loader = HuggingFaceDatasetLoader(DATASET_NAME, PAGE_CONTENT_COLUMN)
data = loader.load()
print(data[0])

page_content='"The Grand Park Hotel exceeded all my expectations. The staff was friendly and accommodating, and the rooms were luxurious and comfortable. The hotel's location in the heart of New York City made it convenient to explore the city's attractions. I would highly recommend this hotel to anyone visiting New York City."' metadata={'hotel': 'Grand Park Hotel', 'city': 'New York City, USA'}


In [4]:
# selecting the most relevant chunk_size and chunk_overlap

for chunk_size in [300, 500, 800, 1200]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_size*0.15)
    chunks = splitter.split_documents(data)
    print(f"chunk_size={chunk_size}: {len(chunks)} chunks")
    print(f"Instance: {chunks[0].page_content}\n")

# optimal value: chunk_size=500; chunk_overlap=75

chunk_size=300: 213 chunks
Instance: "The Grand Park Hotel exceeded all my expectations. The staff was friendly and accommodating, and the rooms were luxurious and comfortable. The hotel's location in the heart of New York City made it convenient to explore the city's attractions. I would highly recommend this hotel to anyone visiting

chunk_size=500: 155 chunks
Instance: "The Grand Park Hotel exceeded all my expectations. The staff was friendly and accommodating, and the rooms were luxurious and comfortable. The hotel's location in the heart of New York City made it convenient to explore the city's attractions. I would highly recommend this hotel to anyone visiting New York City."

chunk_size=800: 105 chunks
Instance: "The Grand Park Hotel exceeded all my expectations. The staff was friendly and accommodating, and the rooms were luxurious and comfortable. The hotel's location in the heart of New York City made it convenient to explore the city's attractions. I would highly recommend t

TESTING TOP_K VALUE

In [5]:
# getting data and vector store

documents = prepare_data(DATASET_NAME, PAGE_CONTENT_COLUMN)
vector_store = create_vector_store(documents)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9416.36it/s]


In [7]:
# selecting the most relevant top_k

for k in [3, 5, 10]:
    retriever = vector_store.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke("Where are the luxury rooms?")
    print(f"k={k}")
    for d in docs:
        print(f"  - {d.page_content[:80]}")

# optimal value: top_k=5

k=3
  - oasis amidst the bustling city. I would highly recommend Urban Oasis Hotel for a
  - "My experience at The Royal Oasis was absolutely delightful. The hotel's regal a
  - "Urban Oasis Hotel provided a wonderful experience during my visit to New Orlean
k=5
  - oasis amidst the bustling city. I would highly recommend Urban Oasis Hotel for a
  - "My experience at The Royal Oasis was absolutely delightful. The hotel's regal a
  - "Urban Oasis Hotel provided a wonderful experience during my visit to New Orlean
  - of Rome's historic sites. I would highly recommend Majestic Plaza Hotel for a re
  - "Urban Oasis Hotel provided a wonderful stay in New Orleans. The hotel's central
k=10
  - oasis amidst the bustling city. I would highly recommend Urban Oasis Hotel for a
  - "My experience at The Royal Oasis was absolutely delightful. The hotel's regal a
  - "Urban Oasis Hotel provided a wonderful experience during my visit to New Orlean
  - of Rome's historic sites. I would highly recomme